In [6]:
import pandas as pd

# ============================================================
# 1. LOAD RAW DATA
# ============================================================
customers = pd.read_csv('olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

# ============================================================
# 2. CREATE CLEAN COPIES
# ============================================================
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

# ============================================================
# 3. CLEAN orders_clean (date fixes + anomaly flagging)
# ============================================================
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

# Anomaly 1: approved_at AFTER carrier_date
approved_after_carrier = orders_clean[
    orders_clean['order_approved_at'].notna() &
    orders_clean['order_delivered_carrier_date'].notna() &
    (orders_clean['order_approved_at'] > orders_clean['order_delivered_carrier_date'])
].copy()
approved_after_carrier['approval_lag_hours'] = (
    approved_after_carrier['order_approved_at'] -
    approved_after_carrier['order_delivered_carrier_date']
).dt.total_seconds() / 3600

# Anomaly 2: carrier_date AFTER customer_date (physically impossible)
carrier_after_customer = orders_clean[
    orders_clean['order_delivered_carrier_date'].notna() &
    orders_clean['order_delivered_customer_date'].notna() &
    (orders_clean['order_delivered_carrier_date'] > orders_clean['order_delivered_customer_date'])
].copy()

# Split lag into minor (<=48h, harmless logging delay) vs severe (>48h)
minor_lag_ids = approved_after_carrier[approved_after_carrier['approval_lag_hours'] <= 48]['order_id']
severe_lag_ids = approved_after_carrier[approved_after_carrier['approval_lag_hours'] > 48]['order_id']

# Build the data_quality_flag column
orders_clean['data_quality_flag'] = 'ok'
orders_clean.loc[orders_clean['order_id'].isin(minor_lag_ids), 'data_quality_flag'] = 'minor_lag'
orders_clean.loc[orders_clean['order_id'].isin(severe_lag_ids), 'data_quality_flag'] = 'severe_anomaly'
orders_clean.loc[orders_clean['order_id'].isin(carrier_after_customer['order_id']), 'data_quality_flag'] = 'severe_anomaly'

print("Data quality flag distribution:")
print(orders_clean['data_quality_flag'].value_counts())

# ============================================================
# 4. BUILD DIMENSION TABLES
# ============================================================

# ---- DIM_DATE ----
all_dates = orders_clean['order_purchase_timestamp'].dt.date.dropna().unique()
dim_date = pd.DataFrame({'full_date': pd.to_datetime(all_dates)})
dim_date['date_id'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['weekday'] = dim_date['full_date'].dt.day_name()
dim_date['quarter'] = dim_date['full_date'].dt.quarter

# ---- DIM_CUSTOMERS ----
dim_customers = customers_clean[['customer_id', 'customer_city', 'customer_state',
                                  'customer_zip_code_prefix']].drop_duplicates()

# ---- DIM_PRODUCTS (with English category names) ----
dim_products = products_clean.merge(
    category_translation_clean, on='product_category_name', how='left'
)[['product_id', 'product_category_name_english', 'product_weight_g']].drop_duplicates()

# ---- DIM_SELLERS ----
dim_sellers = sellers_clean[['seller_id', 'seller_city', 'seller_state']].drop_duplicates()

# ---- DIM_PAYMENT ----
dim_payment = order_payments_clean[['order_id', 'payment_type',
                                     'payment_installments', 'payment_value']].drop_duplicates()

# ---- DIM_ORDER_STATUS ----
dim_order_status = orders_clean[['order_id', 'order_status', 'data_quality_flag']].drop_duplicates()

# ============================================================
# 5. BUILD FACT TABLE (grain: one row per order item)
# ============================================================
fact_order_items = order_items_clean.merge(
    orders_clean[['order_id', 'customer_id', 'order_purchase_timestamp',
                  'order_delivered_customer_date', 'order_estimated_delivery_date',
                  'data_quality_flag']],
    on='order_id', how='left'
)

fact_order_items['date_id'] = fact_order_items['order_purchase_timestamp'].dt.strftime('%Y%m%d')
fact_order_items['date_id'] = fact_order_items['date_id'].astype('Int64')  # nullable int, handles NaT safely

fact_order_items['delivery_time_days'] = (
    fact_order_items['order_delivered_customer_date'] -
    fact_order_items['order_purchase_timestamp']
).dt.days

fact_order_items['is_late'] = (
    fact_order_items['order_delivered_customer_date'] >
    fact_order_items['order_estimated_delivery_date']
)

fact_order_items = fact_order_items[[
    'order_id', 'customer_id', 'product_id', 'seller_id', 'date_id',
    'price', 'freight_value', 'delivery_time_days', 'is_late', 'data_quality_flag'
]]

# ============================================================
# 6. VALIDATE
# ============================================================
print("\n--- Shapes ---")
print(f"dim_date: {dim_date.shape}")
print(f"dim_customers: {dim_customers.shape}")
print(f"dim_products: {dim_products.shape}")
print(f"dim_sellers: {dim_sellers.shape}")
print(f"dim_payment: {dim_payment.shape}")
print(f"dim_order_status: {dim_order_status.shape}")
print(f"fact_order_items: {fact_order_items.shape}")

print("\n--- fact_order_items preview ---")
print(fact_order_items.head())

# ============================================================
# 7. (OPTIONAL) SAVE ALL TABLES TO CSV
# ============================================================
dim_date.to_csv('dim_date.csv', index=False)
dim_customers.to_csv('dim_customers.csv', index=False)
dim_products.to_csv('dim_products.csv', index=False)
dim_sellers.to_csv('dim_sellers.csv', index=False)
dim_payment.to_csv('dim_payment.csv', index=False)
dim_order_status.to_csv('dim_order_status.csv', index=False)
fact_order_items.to_csv('fact_order_items.csv', index=False)
print("\nAll star schema tables saved to CSV.")

Data quality flag distribution:
data_quality_flag
ok                98059
minor_lag          1181
severe_anomaly      201
Name: count, dtype: int64

--- Shapes ---
dim_date: (634, 7)
dim_customers: (99441, 4)
dim_products: (32951, 3)
dim_sellers: (3095, 3)
dim_payment: (103272, 4)
dim_order_status: (99441, 3)
fact_order_items: (112650, 10)

--- fact_order_items preview ---
                           order_id                       customer_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  3ce436f183e68e07877b285a838db11a   
1  00018f77f2f0320c557190d7a144bdd3  f6dd3ec061db4e3987629fe6b26e5cce   
2  000229ec398224ef6ca0657da4fc703e  6489ae5e4333f3693df5ad4372dab6d3   
3  00024acbcdf0a6daa1e931b038114c75  d4eb9395c8c0431ee92fce09860c5a06   
4  00042b26cf59d7ce69dfabb4e55b4fd9  58dbd0b2d70206bf40e62cd34e84d795   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd